Загрузка и подключение библиотек; настройка random_seed для воспроизводимости результатов; подключение гугл диска

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q python-docx
!pip install -q transformers datasets accelerate peft trl bitsandbytes bert-score
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install --upgrade trl

In [ ]:
import os
import re
import gc
import json
import shutil
import torch
import pandas as pd
import numpy as np
from docx import Document
from datasets import Dataset
from tqdm import tqdm
import random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    DataCollatorForSeq2Seq,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModel
)
from bert_score import BERTScorer
import json
import matplotlib.pyplot as plt
import glob

def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

torch.cuda.empty_cache()
gc.collect()

269

Конфигурация

In [ ]:
set_random_seed(27)

class Config:
    DATA_FILES = [
        ('pul_intervyu_1.docx', 'kodirovki_pfi_2023.docx'),
        ('pul_intervyu_2.docx', 'kodirovki_pfi_2024.docx'),
        ('pul_intervyu_3.docx', 'kodirovki_sp_2024.docx'),
        ('pul_intervyu_4.docx', 'kodirovki_pfi_2025.docx'),
    ]

    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = ["q_proj", "v_proj", "o_proj"]

    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    EVAL_BATCH_SIZE = 2
    LEARNING_RATE = 2e-4
    EPOCHS = 2

    GRADIENT_CHECKPOINTING = True
    MASK_PROMPT_LOSS = True
    GROUP_BY_LENGTH = True
    ASSISTANT_TEMPLATE = "<|im_start|>assistant\n"

    OUTPUT_DIR = "./qwen_7b_coding"
    LOGGING_STEPS = 10

Функция для загрузки данных интервью и кодировок (правильных ответов модели)

In [ ]:
def parse_interview_transcript(filename):
    """Парсинг транскриптов интервью"""
    try:
        doc = Document(filename)
    except Exception as e:
        print(f"Ошибка при открытии {filename}: {e}")
        return []

    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]

    if not paragraphs:
        return []

    global_topic = ""
    if not re.match(r'^Интервью\s+\d+', paragraphs[0]):
        global_topic = paragraphs[0]

    interviews = []
    current_interview = None

    for text in paragraphs:
        if text == global_topic:
            continue

        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {
                'interview_num': interview_num,
                'topic': global_topic,
                'transcript': ''
            }
        elif current_interview:
            if current_interview['transcript']:
                current_interview['transcript'] += '\n' + text
            else:
                current_interview['transcript'] = text

    if current_interview:
        interviews.append(current_interview)

    return interviews


def parse_coding(filename):
    """Парсинг кодировок интервью"""
    try:
        doc = Document(filename)
    except Exception as e:
        print(f"Ошибка при открытии {filename}: {e}")
        return []

    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]

    interviews = []
    current_interview = None

    for text in paragraphs:
        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {
                'interview_num': interview_num,
                'coding': ''
            }
        elif current_interview:
            if current_interview['coding']:
                current_interview['coding'] += '\n' + text
            else:
                current_interview['coding'] = text

    if current_interview:
        interviews.append(current_interview)

    return interviews


def load_all_data():
    """Загрузка и объединение всех данных"""
    all_interviews = []

    for trans_file, code_file in Config.DATA_FILES:
        print(f"Обработка {trans_file} и {code_file}...")

        transcripts = parse_interview_transcript(trans_file)
        codings = parse_coding(code_file)

        if not transcripts:
            print(f"Предупреждение: Не удалось загрузить транскрипты из {trans_file}")
            continue

        coding_dict = {c['interview_num']: c['coding'] for c in codings}

        for trans in transcripts:
            interview_num = trans['interview_num']
            coding = coding_dict.get(interview_num, '')

            if coding:
                transcript = trans['transcript']

                all_interviews.append({
                    'id': len(all_interviews),
                    'topic': trans['topic'],
                    'interview_num': interview_num,
                    'transcript': transcript,
                    'coding': coding
                })

    df = pd.DataFrame(all_interviews)
    print(f"Всего загружено интервью с кодировками: {len(df)}")
    return df

Функции для создания промпта для обучения модели

In [ ]:
def tokenize_for_generation(tokenizer, prompt, device):
    """Токенизация"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)
    return inputs


def _find_subsequence(sequence, subsequence):
    for idx in range(len(sequence) - len(subsequence) + 1):
        if sequence[idx: idx + len(subsequence)] == subsequence:
            return idx
    return -1


def mask_prompt_labels(input_ids, response_template_ids):
    """Маскирует system/user в labels, loss только на ответе assistant."""
    labels = list(input_ids)
    start = _find_subsequence(labels, response_template_ids)
    if start == -1:
        return [-100] * len(labels)
    for i in range(start + len(response_template_ids)):
        labels[i] = -100
    return labels


def create_training_prompt(example, tokenizer):
    """Создание промта для обучения с примерами"""

    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Act step by step:

1. Carefully review the output format shown in the example below. Remember that each code block must contain "Общий код" (General code), then Quote, then "Конкретный код" (Specific code). The quote must be verbatim and enclosed in quotation marks.
2. Read the interview transcript and the topic. Identify all fragments (quotes) that relate to the interview topic.
3. Group the quotes by general themes — these will be the "Общий код" (General codes). For each general theme, come up with a short name.
4. Within each general code, identify specific meaning aspects — these will be the "конкретный код" (Specific codes). The names of specific codes should reflect the essence of the quote.
5. Generate the answer strictly following the format from the example. Do not add any explanations, do not write words like 'Step 1', 'Step 2' — only the final blocks of codes and quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Example of a general code related to the topic (pay attention to the structure):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": example['coding']},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return text

Подготовка датасета для обучения: выделение "эталонного" интервью, закодированного лабораторией, а также разделение выборки на тренировочную, валидационную и тестовую

In [ ]:
prompt_example = None

def prepare_dataset(df, tokenizer):
    """Подготовка датасета для обучения"""
    data = []

    global prompt_example
    prompt_example = df.iloc[0]
    df = df.iloc[1:]

    for _, row in df.iterrows():
        if pd.notna(row['coding']) and row['coding'].strip():
            prompt = create_training_prompt(row, tokenizer)
            data.append({'text': prompt})

    return pd.DataFrame(data)

def split_dataset(df):
    """Разделение данных на train/val/test"""
    np.random.seed(42)
    indices = np.random.permutation(len(df))

    n_train = int(0.7 * len(df))
    n_val = int(0.15 * len(df))

    train_df = df.iloc[indices[:n_train]]
    val_df = df.iloc[indices[n_train:n_train + n_val]]
    test_df = df.iloc[indices[n_train + n_val:]]

    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    return train_df, val_df, test_df

Функция для настройки токенизатора

In [ ]:
def setup_tokenizer(model_name):
    """Настройка токенизатора"""
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side="right"
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer

Функция для настройки параметров модели (в частности, ее загрузки) и параметров дообучения

In [ ]:
def setup_model_and_lora(model_name, tokenizer):
    """Настройка модели - квантизация + LoRA"""

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )

    model = prepare_model_for_kbit_training(model)
    if Config.GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
    elif hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=Config.LORA_R,
        lora_alpha=Config.LORA_ALPHA,
        target_modules=Config.LORA_TARGET_MODULES,
        lora_dropout=Config.LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(model, lora_config)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}% of {total_params:,})")

    return model

Функция для оценки модели на тестовых данных

In [ ]:
def evaluate_model(model, tokenizer, test_df, device, num_samples=5):
    """Оценка модели на тестовых данных"""
    model.eval()

    results = []

    for idx, row in test_df.head(num_samples).iterrows():
        text = row['text']
        parts = text.split('<|im_start|>assistant\n')

        if len(parts) > 1:
            prompt = parts[0] + '<|im_start|>assistant\n'
            target = parts[1].split('<|im_end|>')[0].strip()
        else:
            continue

        inputs = tokenize_for_generation(tokenizer, prompt, device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                      skip_special_tokens=True)

        results.append({
            'id': row.get('id', idx),
            'generated': generated[:800],
            'target': target[:500]
        })

    return results

def generate_coding(model, tokenizer, topic, transcript, prompt_example, device=None):
    """Генерация кодировки для нового интервью"""
    if device is None:
        device = next(model.parameters()).device

    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Act step by step:

1. Carefully review the output format shown in the example below. Remember that each code block must contain "Общий код" (General code), then Quote, then "Конкретный код" (Specific code). The quote must be verbatim and enclosed in quotation marks.
2. Read the interview transcript and the topic. Identify all fragments (quotes) that relate to the interview topic.
3. Group the quotes by general themes — these will be the "Общий код" (General codes). For each general theme, come up with a short name.
4. Within each general code, identify specific meaning aspects — these will be the "конкретный код" (Specific codes). The names of specific codes should reflect the essence of the quote.
5. Generate the answer strictly following the format from the example. Do not add any explanations, do not write words like 'Step 1', 'Step 2' — only the final blocks of codes and quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Example of a general code related to the topic (pay attention to the structure):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": f"""Тема интервью:
{topic}

Текст интервью:
{transcript}

Пожалуйста, выполни разметку и кодирование интервью в указанном формате."""},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenize_for_generation(tokenizer, prompt, device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                  skip_special_tokens=True)

    return generated

Функция для обучения модели

In [ ]:
def main():
    df_interviews = load_all_data()

    if len(df_interviews) == 0:
        return None, None, None

    tokenizer = setup_tokenizer(Config.MODEL_NAME)

    df_data = prepare_dataset(df_interviews, tokenizer)
    print(f"Создано {len(df_data)} примеров")

    train_df, val_df, test_df = split_dataset(df_data)
    test_df.to_csv('test_data.csv', index=False)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = setup_model_and_lora(Config.MODEL_NAME, tokenizer)

    response_template_ids = tokenizer.encode(
        Config.ASSISTANT_TEMPLATE, add_special_tokens=False
    )

    def tokenize_function(examples):
        tokenized = tokenizer(
            examples['text'],
        )
        if Config.MASK_PROMPT_LOSS:
            tokenized['labels'] = [
                mask_prompt_labels(ids, response_template_ids)
                for ids in tokenized['input_ids']
            ]
        tokenized['length'] = [len(ids) for ids in tokenized['input_ids']]
        return tokenized

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(
        tokenize_function, batched=True, batch_size=8, remove_columns=['text']
    )
    val_dataset = val_dataset.map(
        tokenize_function, batched=True, batch_size=8, remove_columns=['text']
    )

    if Config.MASK_PROMPT_LOSS:
        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            padding=True,
            label_pad_token_id=-100,
            pad_to_multiple_of=8,
        )
    else:
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
            pad_to_multiple_of=8,
        )

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    print(
        f"batch={Config.BATCH_SIZE}, accum={Config.GRADIENT_ACCUMULATION_STEPS}, "
        f"checkpointing={Config.GRADIENT_CHECKPOINTING}, mask_prompt={Config.MASK_PROMPT_LOSS}, "
        f"dtype={'bf16' if use_bf16 else 'fp16'}"
    )

    training_args = TrainingArguments(
        output_dir=Config.OUTPUT_DIR,
        num_train_epochs=Config.EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        per_device_eval_batch_size=Config.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
        warmup_ratio=0.05,
        learning_rate=Config.LEARNING_RATE,
        bf16=use_bf16,
        fp16=not use_bf16,
        logging_steps=Config.LOGGING_STEPS,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=False,
        report_to="none",
        gradient_checkpointing=Config.GRADIENT_CHECKPOINTING,
        train_sampling_strategy=(
            "group_by_length" if Config.GROUP_BY_LENGTH else "random"
        ),
        length_column_name="length",
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        weight_decay=0.01,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
    )

    effective_batch_size = Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS
    steps_per_epoch = (len(train_dataset) + effective_batch_size - 1) // effective_batch_size
    total_steps = steps_per_epoch * Config.EPOCHS

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    trainer.train()
    trainer.save_model(Config.OUTPUT_DIR)
    tokenizer.save_pretrained(Config.OUTPUT_DIR)

    device = next(model.parameters()).device
    eval_results = evaluate_model(model, tokenizer, test_df, device, num_samples=2)

    if eval_results:
        results_df = pd.DataFrame(eval_results)
        results_df.to_csv('test_evaluation_results.csv', index=False)

        print("\n Пример генерации:")
        example = eval_results[0]
        print(f"Сгенерированный ответ:\n{example['generated'][:800]}...")

    print("Обучение завершено")
    print(f"Модель сохранена в {Config.OUTPUT_DIR}")
    return model, tokenizer, trainer

Обучение модели

In [ ]:
set_random_seed(27)

if __name__ == "__main__":
    model, tokenizer, trainer = main()

    if model:

        print("Тестовый запуск:")
        test_topic = "ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА"
        test_transcript = """
        Интервьюер: Расскажи о своей работе.
        Информант: Мне очень нравится моя работа. У нас отличный коллектив,
        всегда можно обратиться за помощью. И есть возможность развиваться
        профессионально, компания оплачивает курсы.
        """
        result = generate_coding(model, tokenizer, test_topic, test_transcript, prompt_example)
        print(f"\nРезультат:\n{result}")

Обработка pul_intervyu_1.docx и kodirovki_pfi_2023.docx...
Обработка pul_intervyu_2.docx и kodirovki_pfi_2024.docx...
Обработка pul_intervyu_3.docx и kodirovki_sp_2024.docx...
Обработка pul_intervyu_4.docx и kodirovki_pfi_2025.docx...
Всего загружено интервью с кодировками: 150


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


batch=1, accum=8, checkpointing=True, mask_prompt=True, dtype=bf16


Step,Training Loss
10,2.223145
20,2.031078


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



 Пример генерации:
Сгенерированный ответ:
**Общий код 1: Проблемы социальной интеграции и миграционных вызовов**
"Да, вот, я вот, наверное, не много людей знаю, которые бы могли сказать, что они всё время чувствуют себя комфортно." - **Не комфортные отношения с местными жителями (конкретный код)**
"Ну, вот, я всё время слышал, что люди говорят, "ты русский, а мы немец", или "ты русский, а мы украинец". Это всё время, когда я чувствую, что я не местный." - **Отсутствие чувства принадлежности (конкретный код)**
"Ну, вот, я всё время слышал, что люди говорят, "ты русский, а мы немец", или "ты русский, а мы украинец". Это всё время, когда я чувствую, что я не местный." - **Отсутствие чувства принадлежности (конкретный код)**
"Ну, вот, я всё время слышал, что люди говорят, "ты русский, а мы немец", или "ты русский, а мы украинец". Это ...
Обучение завершено
Модель сохранена в ./qwen_7b_coding
Тестовый запуск:

Результат:
**Общий код 1: Настраивание на работу и поиск баланса**

"Я считаю, ч

Видно, что модель зацикливается в ответах, повторяет цитаты. Исправим это с помощью новой функции генерации, которая штрафует за повторения

In [ ]:
def save_model_complete(model, tokenizer, output_dir="./qwen_coding_3b_model"):
    """Сохраняет только LoRA-адаптер, без повторного сохранения модели"""
    os.makedirs(output_dir, exist_ok=True)

    src = Config.OUTPUT_DIR
    adapter_src = None
    if os.path.exists(os.path.join(src, "adapter_config.json")):
        adapter_src = src
    elif os.path.isdir(src):
        for name in sorted(os.listdir(src), reverse=True):
            path = os.path.join(src, name)
            if name.startswith("checkpoint-") and os.path.isdir(path):
                if os.path.exists(os.path.join(path, "adapter_config.json")):
                    adapter_src = path
                    break

    if adapter_src is None:
        from peft import PeftModel
        if isinstance(model, PeftModel):
            model.save_pretrained(output_dir, safe_serialization=True)
        else:
            raise FileNotFoundError(
                f"adapter_config.json не найден в {src}. Сначала запустите обучение."
            )
    else:
        for fname in os.listdir(adapter_src):
            if fname.startswith("adapter") or fname == "README.md":
                shutil.copy2(os.path.join(adapter_src, fname), output_dir)

    tokenizer.save_pretrained(output_dir)

    config = {
        "base_model": Config.MODEL_NAME,
        "lora_r": Config.LORA_R,
        "lora_alpha": Config.LORA_ALPHA,
        "model_type": "qwen_lora"
    }

    with open(f"{output_dir}/model_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    if not os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        raise FileNotFoundError(f"adapter_config.json не сохранился в {output_dir}")

    print(f"LoRA-адаптер сохранён в {output_dir}")
    print("Файлы:", os.listdir(output_dir))
    return output_dir

In [ ]:
save_model_complete(model, tokenizer, "/content/drive/MyDrive/qwen_3b_model_lora_0")

LoRA-адаптер сохранён в /content/drive/MyDrive/qwen_3b_model_lora_0
Файлы: ['adapter_model.safetensors', 'README.md', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'model_config.json']


'/content/drive/MyDrive/qwen_3b_model_lora_0'

In [ ]:
from peft import PeftModel

In [ ]:
set_random_seed(27)

def clear_cuda_cache():
    """Очистка кэша CUDA памяти"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        return allocated, reserved
    return 0, 0

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

def find_adapter_dir(*paths):
    for path in paths:
        if not os.path.isdir(path):
            continue
        if os.path.exists(os.path.join(path, "adapter_config.json")):
            return path
        for name in sorted(os.listdir(path), reverse=True):
            sub = os.path.join(path, name)
            if name.startswith("checkpoint-") and os.path.isdir(sub):
                if os.path.exists(os.path.join(sub, "adapter_config.json")):
                    return sub
    raise FileNotFoundError(
        "adapter_config.json не найден"
        f"Проверены пути: {list(paths)}"
    )

ADAPTER_PATH = find_adapter_dir(
    "/content/drive/MyDrive/qwen_3b_model_lora_0",
    Config.OUTPUT_DIR,
    "./qwen_7b_coding",
)
print(f"Загрузка адаптера из: {ADAPTER_PATH}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

clear_cuda_cache()


try:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
except Exception as e:
    model = PeftModel.from_pretrained(
        model,
        ADAPTER_PATH,
        is_trainable=False,
        config=None
    )

model.eval()
clear_cuda_cache()


def generate_without_repetition(prompt):
    """Генерация с очисткой памяти и удалением повторов"""
    clear_cuda_cache()
    inputs = tokenize_for_generation(tokenizer, prompt, model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            repetition_penalty=1.5,
            no_repeat_ngram_size=5,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    lines = generated.strip().split('\n')
    seen = set()
    unique_lines = []

    for line in lines:
        if not line.strip():
            continue
        key = line[:50]
        if key not in seen:
            seen.add(key)
            unique_lines.append(line)
        else:
            break

    clear_cuda_cache()
    return '\n'.join(unique_lines[:15])

Загрузка адаптера из: /content/drive/MyDrive/qwen_3b_model_lora_0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
test_df = pd.read_csv('test_data.csv')
print(f"Тестовых примеров: {len(test_df)}")

for idx in range(min(2, len(test_df))):
    print(f"\n Пример {idx + 1}")
    full_text = test_df.iloc[idx]['text']
    parts = full_text.split('<|im_start|>assistant\n')
    prompt = parts[0] + '<|im_start|>assistant\n'

    print("\n Предсказание модели:")
    generated = generate_without_repetition(prompt)
    print(generated)
    clear_cuda_cache()

Тестовых примеров: 23

 Пример 1

 Предсказание модели:
*Отсутствуют конкретные цели/желающие ответить*
Необходимость дополнений перед дальнейшей работосозданием запроса данных.*  
Перевести вопросы обратного характера необходимо через "скептики", возможно добавление определённых фрагментов.*
Пример:
*"Как тебе такое отношение?" 
-- *Проективность*: "...что люди могут быть предубежденными..." -- ***Стремления преобразовать общество*** ("...можете попробывать изменять ситуацию вокруг вас...")
"...прямые действия..."
"-- ...можно сделать шаг за собой..."
--- 
---
### Ответ интуфронтального типа без специй / контрастированный типичному образцу реагирования молодых людей...

 Пример 2

 Предсказание модели:
**Общий код 1: Образования молодежного поколения**
"... Я консультироваться могу только через интернет..." - **Отсутствие довериия ко всем другим источникам информации кроме сети Интернет (код №6).**
* * *
**Общей код 2: Связанные факторы влияния*
*"А ведь раньше люди общались напрямую!

Сохранение модели на Диск

In [ ]:
save_model_complete(model, tokenizer, "/content/drive/MyDrive/qwen_3b_model_lora")

LoRA-адаптер сохранён в /content/drive/MyDrive/qwen_3b_model_lora
Файлы: ['adapter_model.safetensors', 'tokenizer.json', 'chat_template.jinja', 'tokenizer_config.json', 'model_config.json', 'adapter_config.json', 'README.md']


'/content/drive/MyDrive/qwen_3b_model_lora'